# ShopSphere Repurchase Prediction: Classical Machine Learning (LightGBM)
**Course:** IT2226 AI Innovation and Deep Learning Project  
**Target:** Customer Repurchase in October 2024 (`target = 1` vs `0`)  
**Database:** `shopsphere.db` (Relational SQLite schema with verified Foreign Keys)  
**Architecture:** Direct SQL feature extraction, class-imbalance reweighting, 5-Fold Stratified Cross-Validation, decision threshold tuning, and holdout inference.

## 1. Environment Setup & File Availability
Install LightGBM, upload `shopsphere.db` if running on Google Colab, and import analytical packages.

In [ ]:
# Install required packages
!pip install lightgbm scikit-learn matplotlib seaborn -q

import os
import time
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    classification_report,
    f1_score,
    roc_curve,
    precision_recall_curve,
    confusion_matrix
)
import lightgbm as lgb

# Check for database file
DB_PATH = 'shopsphere.db'
if not os.path.exists(DB_PATH):
    print('Database shopsphere.db not found. Please upload it.')
    try:
        from google.colab import files
        uploaded = files.upload()
    except Exception as e:
        print('Not in Colab or upload skipped:', e)
else:
    size_mb = os.path.getsize(DB_PATH) / (1024 * 1024)
    print(f'Database detected: {DB_PATH} ({size_mb:.1f} MB)')
# Enforce fixed seed globally for deterministic results
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
import random
random.seed(RANDOM_SEED)


## 2. On-Demand SQL Customer 360 Feature Extraction
To eliminate row explosion and memory bloat, we aggregate child tables (transactions, sessions, deliveries, support tickets, and retention contact history) to the `customer_id` level directly inside SQLite before joining.

In [ ]:
t0 = time.time()
conn = sqlite3.connect(DB_PATH)

feature_query = '''
WITH cust_rfm AS (
    SELECT 
        customer_id,
        COUNT(transaction_id) AS total_orders,
        COALESCE(SUM(order_value), 0.0) AS total_spend_usd,
        COALESCE(AVG(order_value), 0.0) AS avg_order_value,
        COALESCE(MAX(order_value), 0.0) AS max_order_value,
        COALESCE(SUM(items_count), 0) AS total_items_bought,
        COALESCE(SUM(discount_applied), 0) AS discounted_orders_count,
        COALESCE(SUM(high_value_flag), 0) AS high_value_orders_count,
        COALESCE(SUM(CASE WHEN transaction_timestamp >= '2024-09-01' THEN 1 ELSE 0 END), 0) AS orders_last_30d,
        COALESCE(SUM(CASE WHEN transaction_timestamp >= '2024-07-01' THEN 1 ELSE 0 END), 0) AS orders_last_90d,
        ROUND(JULIANDAY('2024-09-30') - JULIANDAY(MAX(transaction_timestamp)), 1) AS recency_days
    FROM transactions
    GROUP BY customer_id
),
cust_sessions AS (
    SELECT 
        customer_id,
        COUNT(session_id) AS total_sessions,
        COALESCE(SUM(pages_viewed), 0) AS total_pages_viewed,
        COALESCE(SUM(cart_additions), 0) AS total_cart_additions,
        COALESCE(SUM(bounce_flag), 0) AS total_bounced_sessions,
        COALESCE(AVG(session_duration_capped), 0.0) AS avg_session_duration_s,
        COALESCE(SUM(CASE WHEN session_timestamp >= '2024-09-01' THEN 1 ELSE 0 END), 0) AS sessions_last_30d,
        COALESCE(SUM(CASE WHEN session_timestamp >= '2024-07-01' THEN 1 ELSE 0 END), 0) AS sessions_last_90d,
        ROUND(JULIANDAY('2024-09-30') - JULIANDAY(MAX(session_timestamp)), 1) AS days_since_last_session
    FROM sessions
    GROUP BY customer_id
),
cust_deliveries AS (
    SELECT 
        t.customer_id,
        COUNT(d.transaction_id) AS total_deliveries,
        COALESCE(SUM(CASE WHEN d.delivery_status = 'Late' THEN 1 ELSE 0 END), 0) AS late_deliveries_count,
        COALESCE(SUM(CASE WHEN d.delivery_status = 'Lost' THEN 1 ELSE 0 END), 0) AS lost_packages_count,
        COALESCE(AVG(d.delay_days), 0.0) AS avg_delay_days
    FROM transactions t
    JOIN deliveries d ON t.transaction_id = d.transaction_id
    GROUP BY t.customer_id
),
cust_tickets AS (
    SELECT 
        customer_id,
        COUNT(ticket_id) AS total_support_tickets,
        COALESCE(SUM(CASE WHEN status = 'Open' THEN 1 ELSE 0 END), 0) AS open_tickets_count,
        COALESCE(SUM(CASE WHEN status = 'Open' AND priority IN ('High', 'Urgent') THEN 1 ELSE 0 END), 0) AS high_priority_open_tickets,
        AVG(csat_score) AS avg_csat_score
    FROM support_tickets
    GROUP BY customer_id
),
cust_retention AS (
    SELECT 
        customer_id,
        COUNT(contact_id) AS retention_contacts_count,
        COALESCE(SUM(redeemed), 0) AS retention_redeemed_count,
        COALESCE(SUM(purchased_within_14d), 0) AS retention_purchased_count
    FROM retention_contact_history
    GROUP BY customer_id
)
SELECT 
    c.customer_id,
    c.age,
    c.age_is_missing,
    c.country_code,
    c.customer_tier,
    c.credit_limit,
    c.loyalty_score,
    c.email_open_rate,
    c.email_open_rate_is_missing,
    c.avg_review_score,
    ROUND(JULIANDAY('2024-09-30') - JULIANDAY(c.signup_date), 1) AS account_age_days,
    COALESCE(p.email_opt_in, 0) AS email_opt_in,
    COALESCE(p.sms_opt_in, 0) AS sms_opt_in,
    COALESCE(p.do_not_contact, 0) AS do_not_contact,
    COALESCE(p.preferred_channel, 'unknown') AS preferred_channel,
    COALESCE(rfm.total_orders, 0) AS total_orders,
    COALESCE(rfm.total_spend_usd, 0.0) AS total_spend_usd,
    COALESCE(rfm.avg_order_value, 0.0) AS avg_order_value,
    COALESCE(rfm.max_order_value, 0.0) AS max_order_value,
    COALESCE(rfm.total_items_bought, 0) AS total_items_bought,
    COALESCE(rfm.discounted_orders_count, 0) AS discounted_orders_count,
    COALESCE(rfm.high_value_orders_count, 0) AS high_value_orders_count,
    COALESCE(rfm.orders_last_30d, 0) AS orders_last_30d,
    COALESCE(rfm.orders_last_90d, 0) AS orders_last_90d,
    rfm.recency_days,
    COALESCE(s.total_sessions, 0) AS total_sessions,
    COALESCE(s.total_pages_viewed, 0) AS total_pages_viewed,
    COALESCE(s.total_cart_additions, 0) AS total_cart_additions,
    COALESCE(s.total_bounced_sessions, 0) AS total_bounced_sessions,
    COALESCE(s.avg_session_duration_s, 0.0) AS avg_session_duration_s,
    COALESCE(s.sessions_last_30d, 0) AS sessions_last_30d,
    COALESCE(s.sessions_last_90d, 0) AS sessions_last_90d,
    s.days_since_last_session,
    COALESCE(d.total_deliveries, 0) AS total_deliveries,
    COALESCE(d.late_deliveries_count, 0) AS late_deliveries_count,
    COALESCE(d.lost_packages_count, 0) AS lost_packages_count,
    COALESCE(d.avg_delay_days, 0.0) AS avg_delay_days,
    COALESCE(t.total_support_tickets, 0) AS total_support_tickets,
    COALESCE(t.open_tickets_count, 0) AS open_tickets_count,
    COALESCE(t.high_priority_open_tickets, 0) AS high_priority_open_tickets,
    t.avg_csat_score,
    COALESCE(r.retention_contacts_count, 0) AS retention_contacts_count,
    COALESCE(r.retention_redeemed_count, 0) AS retention_redeemed_count,
    COALESCE(r.retention_purchased_count, 0) AS retention_purchased_count,
    tr.target,
    CASE WHEN te.customer_id IS NOT NULL THEN 1 ELSE 0 END AS is_test
FROM customers c
LEFT JOIN customer_preferences p ON c.customer_id = p.customer_id
LEFT JOIN cust_rfm rfm ON c.customer_id = rfm.customer_id
LEFT JOIN cust_sessions s ON c.customer_id = s.customer_id
LEFT JOIN cust_deliveries d ON c.customer_id = d.customer_id
LEFT JOIN cust_tickets t ON c.customer_id = t.customer_id
LEFT JOIN cust_retention r ON c.customer_id = r.customer_id
LEFT JOIN train tr ON c.customer_id = tr.customer_id
LEFT JOIN test te ON c.customer_id = te.customer_id;
'''

df = pd.read_sql_query(feature_query, conn)
conn.close()
print(f'Extracted {len(df):,} customer records in {time.time() - t0:.2f} seconds.')
df.head()

## 3. Behavioral Feature Engineering & Type Casting
We compute key behavioral ratios: cart abandonment rate, discount reliance, bounce rate, and inactivity flags.

In [ ]:
# Derived domain ratios
df['discount_order_ratio'] = np.where(df['total_orders'] > 0, df['discounted_orders_count'] / df['total_orders'], 0.0)
df['bounce_rate'] = np.where(df['total_sessions'] > 0, df['total_bounced_sessions'] / df['total_sessions'], 0.0)
df['cart_abandonment_ratio'] = np.where(
    df['total_cart_additions'] > 0,
    np.clip(1.0 - (df['total_orders'] / df['total_cart_additions']), 0.0, 1.0),
    0.0
)
df['is_inactive_90d'] = ((df['orders_last_90d'] == 0) & (df['sessions_last_90d'] == 0)).astype(int)

# Set categorical dtypes for LightGBM optimal branching
CATEGORICAL_COLS = ['country_code', 'customer_tier', 'preferred_channel']
for col in CATEGORICAL_COLS:
    df[col] = df[col].astype('category')

EXCLUDE_COLS = ['customer_id', 'target', 'is_test']
FEATURE_COLS = [c for c in df.columns if c not in EXCLUDE_COLS]

train_df = df[df['is_test'] == 0].copy()
test_df = df[df['is_test'] == 1].copy()

X = train_df[FEATURE_COLS]
y = train_df['target'].astype(int)
X_test = test_df[FEATURE_COLS]

print(f'Engineered features: {len(FEATURE_COLS)}')
print(f'Train customers: {len(X):,} | Repurchase Rate: {y.mean():.2%}')
print(f'Holdout test customers: {len(X_test):,}')

## Step 3: Model Training (80/20 Train/Test Holdout Split, seed=42)
We partition the 64,000 labeled training dataset into an **80% training set (51,200 rows)** and a **20% holdout test set (12,800 rows)** with stratification and fixed `random_state=42`.


In [ ]:
# 80/20 Stratified Holdout Split
X_train_80, X_test_20, y_train_80, y_test_20 = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

print(f"Training Set (80%): {len(X_train_80):,} samples (Positives: {y_train_80.sum():,})")
print(f"Holdout Test (20%): {len(X_test_20):,} samples (Positives: {y_test_20.sum():,})")

scale_pos_weight = (len(y_train_80) - y_train_80.sum()) / y_train_80.sum()

params = {
    'objective': 'binary',
    'metric': ['auc', 'binary_logloss'],
    'boosting_type': 'gbdt',
    'learning_rate': 0.03,
    'num_leaves': 31,
    'max_depth': -1,
    'min_child_samples': 30,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'scale_pos_weight': scale_pos_weight,
    'random_state': 42,
    'feature_fraction_seed': 42,
    'bagging_seed': 42,
    'data_random_seed': 42,
    'extra_seed': 42,
    'n_jobs': -1,
    'verbose': -1
}

trn_data = lgb.Dataset(X_train_80, label=y_train_80, categorical_feature=CATEGORICAL_COLS)
val_data = lgb.Dataset(X_test_20, label=y_test_20, reference=trn_data, categorical_feature=CATEGORICAL_COLS)

print('=' * 65)
print('STARTING LIGHTGBM TRAINING ON 80% HOLDOUT (seed=42)')
print('=' * 65)

model = lgb.train(
    params,
    trn_data,
    num_boost_round=1200,
    valid_sets=[trn_data, val_data],
    valid_names=['train', 'val_holdout'],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50, verbose=False),
        lgb.log_evaluation(period=0)
    ]
)

# Predict on 20% holdout test set
val_pred_probs = model.predict(X_test_20, num_iteration=model.best_iteration)
test_preds = model.predict(X_test, num_iteration=model.best_iteration)
feature_importances = model.feature_importance(importance_type='gain')

holdout_auc = roc_auc_score(y_test_20, val_pred_probs)
holdout_pr_auc = average_precision_score(y_test_20, val_pred_probs)

print(f'Training Completed at Best Iteration: {model.best_iteration}')
print(f'20% Holdout ROC-AUC : {holdout_auc:.5f}')
print(f'20% Holdout PR-AUC  : {holdout_pr_auc:.5f}')


## 5. Out-Of-Fold Evaluation & Threshold Tuning
We evaluate Out-Of-Fold discrimination and sweep thresholds to optimize the F1-Score.

In [ ]:
# Model Evaluation & Threshold Tuning on 20% Holdout Test Set
best_th = 0.5
best_f1 = 0.0

for th in np.arange(0.2, 0.8, 0.01):
    f1 = f1_score(y_test_20, (val_pred_probs >= th).astype(int))
    if f1 > best_f1:
        best_f1 = f1
        best_th = th

from sklearn.metrics import recall_score, precision_score

y_pred_holdout = (val_pred_probs >= best_th).astype(int)
rec_score = recall_score(y_test_20, y_pred_holdout)
prec_score = precision_score(y_test_20, y_pred_holdout)
acc_score = np.mean(y_pred_holdout == y_test_20)

print('=' * 65)
print('FINAL 20% HOLDOUT EVALUATION (LightGBM Baseline, seed=42):')
print(f'  ROC-AUC:   {holdout_auc:.4f}')
print(f'  PR-AUC:    {holdout_pr_auc:.4f}')
print(f'  Best F1:   {best_f1:.4f} (at threshold {best_th:.2f})')
print(f'  Recall:    {rec_score:.4f} ({rec_score*100:.2f}%)')
print(f'  Precision: {prec_score:.4f} ({prec_score*100:.2f}%)')
print(f'  Accuracy:  {acc_score:.4f} ({acc_score*100:.2f}%)')
print('=' * 65)

print('\nClassification Report on 20% Holdout:')
print(classification_report(y_test_20, y_pred_holdout, digits=4))


## 6. Visual Diagnostics & Feature Importance
Plotting ROC Curve, Precision-Recall Curve, and ranking the top predictive drivers.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC Curve
fpr, tpr, _ = roc_curve(y, oof_preds)
axes[0].plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC (AUC = {overall_auc:.4f})')
axes[0].plot([0, 1], [0, 1], color='navy', lw=1.5, linestyle='--')
axes[0].set_title('Out-Of-Fold ROC Curve')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].legend(loc='lower right')
axes[0].grid(True, alpha=0.3)

# Precision-Recall Curve
prec, rec, _ = precision_recall_curve(y, oof_preds)
axes[1].plot(rec, prec, color='forestgreen', lw=2, label=f'PR (AUC = {overall_pr_auc:.4f})')
axes[1].set_title('Out-Of-Fold Precision-Recall Curve')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].legend(loc='lower left')
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Feature Importance Plot
fi_df = pd.DataFrame({
    'feature': FEATURE_COLS,
    'gain': feature_importances
}).sort_values(by='gain', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=fi_df.head(15), x='gain', y='feature', palette='viridis')
plt.title('Top 15 Most Predictive Features by Gain')
plt.xlabel('Split Gain')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

print(fi_df.head(10).to_string(index=False))

## 7. Submission Export & Download
Export holdout customer predictions formatted to match `sample_submission.csv`.

In [ ]:
sub_df = pd.DataFrame({
    'customer_id': test_df['customer_id'].astype(int),
    'target': (test_preds >= best_thresh).astype(int)
})

sub_path = 'submission_lightgbm.csv'
sub_df.to_csv(sub_path, index=False)
print(f'Submission saved to {sub_path}')
print(f'Predicted repurchase rate: {sub_df["target"].mean():.2%} ({sub_df["target"].sum():,} / {len(sub_df):,})')

# Auto download if running in Google Colab
try:
    from google.colab import files
    files.download(sub_path)
except Exception:
    pass

## 8. Interactive Customer Predictor (Test Out the Model Live)
Test individual customers by `customer_id` or test hypothetical custom profiles to see real-time repurchase probability and business action recommendations.

In [ ]:
# --- 1. Test Existing Customer by ID ---
def test_customer(customer_id):
    cust_row = df[df['customer_id'] == customer_id]
    if cust_row.empty:
        print(f'Customer {customer_id} not found in database.')
        return
    
    features = cust_row[FEATURE_COLS]
    prob = float(model.predict(features)[0])
    is_repurchase = prob >= best_thresh
    
    # Business Strategy Categorization
    if prob >= 0.70:
        segment = 'Organic Buyer (High Loyalty)'
        action = 'DO NOT DISCOUNT. Customer will buy organically; save marketing margin.'
    elif prob >= best_thresh:
        segment = 'Persuadable Lead (Ideal Target)'
        action = 'SEND RETENTION OFFER (e.g. 10% voucher / free gift) to guarantee conversion.'
    elif prob >= 0.15:
        segment = 'At-Risk Customer'
        action = 'Engage with low-cost channel (Email/SMS newsletter or top trending products).'
    else:
        segment = 'Dormant / Lost'
        action = 'Check support tickets for past bad experiences or suppress from budget.'
    
    print('=' * 65)
    print(f'CUSTOMER 360 PREDICTION: ID {customer_id}')
    print('=' * 65)
    print(f'Tier: {cust_row["customer_tier"].values[0]} | Country: {cust_row["country_code"].values[0]} | Loyalty Score: {cust_row["loyalty_score"].values[0]:.2f}')
    print(f'Total Spend: ${cust_row["total_spend_usd"].values[0]:,.2f} across {int(cust_row["total_orders"].values[0])} orders')
    print(f'Days Since Last Visit: {cust_row["days_since_last_session"].values[0]} | Sessions (90d): {int(cust_row["sessions_last_90d"].values[0])}')
    print(f'Open Tickets: {int(cust_row["open_tickets_count"].values[0])} | Late Deliveries: {int(cust_row["late_deliveries_count"].values[0])}')
    print('-' * 65)
    print(f'Predicted Repurchase Probability : {prob * 100:.2f}%')
    print(f'Model Classification Decision    : {"BUY (1)" if is_repurchase else "NO BUY (0)"}')
    print(f'Targeting Segment               : {segment}')
    print(f'Recommended Action              : {action}')
    print('=' * 65)

# Sample Tests:
# Try an active high-frequency customer vs a dormant customer
sample_ids = train_df['customer_id'].head(5).tolist()
print('Testing first sample customer:')
test_customer(sample_ids[0])